# Hierarchical Two-Layer Fragments Matching with Muti-Dimensional Macro States

This notebook extends the hierarchical price-macro framework by replacing 
the single volatility proxy with a structure macroeconomic state vector
(market return, VIX, yield curve slope, relative strength), and examines
its effect on conditional posterior concentration.

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt

In [5]:
base_path = "/Users/jiayaozhang/project_aapl/data/"

start_date = "1990-01-02"

aapl = pd.read_csv(base_path + "aapl_us_d.csv", sep=";")
aapl["Date"] = pd.to_datetime(aapl["Date"])
aapl = aapl.set_index("Date")

price = aapl["Close"].loc[start_date:]
price_logret = np.log(price / price.shift(1))

def create_price_fragments(series, window=10):
    frags = []
    for i in range(len(series) - window + 1):
        frag = series.iloc[i:i+window].values
        frag = (frag - frag.mean())/(frag.std() + 1e-8)
        frags.append(frag)
    return np.array(frags)

window_size = 10
price_fragments = create_price_fragments(price, window_size)

print("Price fragments shape:", price_fragments.shape)

Price fragments shape: (9055, 10)


# --- Load macro data (Stooq / FRED) ---

In [6]:
print(aapl.index.min(), aapl.index.max())

1984-09-07 00:00:00 2025-12-29 00:00:00


In [7]:
spy = pd.read_csv(base_path + "SPY.csv")
spy["Date"] = pd.to_datetime(spy["Date"])
spy = spy.set_index("Date")["Close"]

qqq = pd.read_csv(base_path + "QQQ.csv")
qqq["Date"] = pd.to_datetime(qqq["Date"])
qqq = qqq.set_index("Date")["Close"]

vix = pd.read_csv(base_path + "vix.csv")
vix["observation_date"] = pd.to_datetime(vix["observation_date"])
vix = vix.set_index("observation_date")["VIXCLS"]

print(spy.head())
print(qqq.head())
print(vix.head())

Date
2005-02-25    93.6948
2005-02-28    93.0544
2005-03-01    93.5386
2005-03-02    93.4927
2005-03-03    93.5301
Name: Close, dtype: float64
Date
1999-03-10    43.2932
1999-03-11    43.4977
1999-03-12    42.4471
1999-03-15    43.6631
1999-03-16    44.0379
Name: Close, dtype: float64
observation_date
1990-01-02    17.24
1990-01-03    18.19
1990-01-04    19.22
1990-01-05    20.11
1990-01-08    20.26
Name: VIXCLS, dtype: float64


Step 1: 统一样本起点(以VIX为准：1990-01-02)

In [8]:
start_date = "1990-01-02"

price = price.loc[start_date:]
price_logret = price_logret.loc[start_date:]
spy = spy.loc[start_date:]
qqq = qqq.loc[start_date:]
vix = vix.loc[start_date:]

Step 2: 构造三维宏观特征(market, risk, relative strength)

In [9]:
macro_df = pd.DataFrame({
    "AAPL": price,
    "SPY": spy,
    "QQQ": qqq,
    "VIX": vix
}).dropna()

       (1)市场方向

In [10]:
macro_df["market_ret"] = np.log(macro_df["SPY"]/macro_df["SPY"].shift(1))

      （2）风险情绪

In [11]:
macro_df["vix_level"] = macro_df["VIX"]

      （3）相对强弱（苹果 vs 纳指100）

In [12]:
macro_df["rel_strength"] = np.log(macro_df["AAPL"]/macro_df["QQQ"])

In [13]:
macro_df = macro_df.dropna()

print(type(macro_df))
print(macro_df.head())

<class 'pandas.core.frame.DataFrame'>
               AAPL      SPY      QQQ    VIX  market_ret  vix_level  \
2005-02-28  1.35003  93.0544  31.8690  12.08   -0.006858      12.08   
2005-03-01  1.33928  93.5386  32.2166  12.04    0.005190      12.04   
2005-03-02  1.32776  93.4927  32.2066  12.50   -0.000491      12.50   
2005-03-03  1.25770  93.5301  31.9413  12.93    0.000400      12.93   
2005-03-04  1.28825  94.7073  32.1292  11.94    0.012508      11.94   

            rel_strength  
2005-02-28     -3.161507  
2005-03-01     -3.180350  
2005-03-02     -3.188678  
2005-03-03     -3.234615  
2005-03-04     -3.216481  


此时每一天都有一个3维宏观组状态向量：
Mt=(market trend, risk level, relative strength)

In [14]:
aligned_price = price.loc[macro_df.index]

price_fragments = create_price_fragments(aligned_price, window_size)

price_logret = np.log(aligned_price/aligned_price.shift(1))

Step 3: 窗口平均--》片段级宏观状态（与price_fragments对齐）

In [15]:
def build_macro_states(df, window):
    states = []
    for i in range(len(df) - window + 1):
        window_state = df.iloc[i:i+window].mean().values
        states.append(window_state)
    return np.array(states)


macro_features = macro_df[["market_ret", "vix_level", "rel_strength"]].copy()

# Standardize each macro feature
macro_features = (macro_features - macro_features.mean()) / macro_features.std(ddof=0)

# Build one 3-dimensional macro state per 10-day window
macro_states = build_macro_states(macro_features, window_size)

print("Macro states shape:", macro_states.shape)
print("Price fragments shape:", price_fragments.shape)

Macro states shape: (5233, 3)
Price fragments shape: (5233, 10)


In [16]:
from sklearn.metrics.pairwise import euclidean_distances

target_index = 2000

# Layer 1: rank all historical fragments by price similarity
target_price = price_fragments[target_index:target_index+1]

price_dist = euclidean_distances(
    target_price, price_fragments
).ravel()

price_sim = 1 / (1 + price_dist)

price_scores = price_sim.copy()
price_scores[target_index] = -np.inf

# Price-only baseline: top 30 by price
price_only_topk = np.argsort(price_scores)[-30:][::-1]

# Two-layer method: first retain top 100 by price
price_top100 = np.argsort(price_scores)[-100:][::-1]

# Layer 2: standardize macro features above, then rank
# the 100 price candidates by macro similarity
target_macro = macro_states[target_index:target_index+1]

macro_dist = euclidean_distances(
    target_macro, macro_states
).ravel()

macro_sim = 1 / (1 + macro_dist)

macro_scores_candidates = macro_sim[price_top100]

# Keep the 30 most macro-similar candidates
macro_order = np.argsort(macro_scores_candidates)[::-1]
two_layer_topk = price_top100[macro_order[:30]]

In [23]:
H=5
future_returns = []

for i in range(len(aligned_price) - window_size + 1):
    end = i + window_size
    if end + H < len(aligned_price):
        future_returns.append(
            np.log(aligned_price.iloc[end+H]/aligned_price.iloc[end])
        )
    else:
        future_returns.append(np.nan)

future_returns = np.array(future_returns)

def get_posterior(idxs):
    vals = future_returns[idxs]
    return vals[~np.isnan(vals)]

posterior_price = get_posterior(price_only_topk)
posterior_two_layer = get_posterior(two_layer_topk)

def iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)

print("5-day forward return")
print("Price-only variance:", np.var(posterior_price))
print("Two-layer variance:", np.var(posterior_two_layer))
print("Price-only IQR:", iqr(posterior_price))
print("Two-layer IQR:", iqr(posterior_two_layer))

5-day forward return
Price-only variance: 0.0025314608890602006
Two-layer variance: 0.0013593287674826596
Price-only IQR: 0.06735098746673655
Two-layer IQR: 0.04607788422345103


## Extension: Conditioning on Future Risk (20-day Realized Volatility)
I replaced the short-horizon return outcome with medium-horizon realized volatility
to test whether macro states primarily constrain risk regimes rather than directional returns.

# === Future 20-day realized volatility as risk outcome ===

In [24]:
H = 20
future_vol = []

log_returns = np.log(aligned_price / aligned_price.shift(1))

for i in range(len(aligned_price) - window_size + 1):
    end = i + window_size

    if end + H < len(aligned_price):
        window_ret = log_returns.iloc[end+1:end+H+1]
        future_vol.append(window_ret.std(ddof=0))
    else:
        future_vol.append(np.nan)

future_vol = np.array(future_vol)

def get_posterior(idxs):
    vals = future_vol[idxs]
    return vals[~np.isnan(vals)]

posterior_price = get_posterior(price_only_topk)
posterior_two_layer = get_posterior(two_layer_topk)

print("20-day realized volatility")
print("Price-only variance:", np.var(posterior_price))
print("Two-layer variance:", np.var(posterior_two_layer))
print("Price-only IQR:", iqr(posterior_price))
print("Two-layer IQR:", iqr(posterior_two_layer))

20-day realized volatility
Price-only variance: 0.00016113471027890072
Two-layer variance: 2.9475414447942702e-05
Price-only IQR: 0.010443297439699854
Two-layer IQR: 0.0049826362832512735
